<a href="https://colab.research.google.com/github/shp2026/some-r-set-training/blob/main/notebook/Some_R_Set_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gymnasium stable-baselines3 sb3-contrib

# Some-R-Set Deck

In [ ]:
"""
Card deck structure for Some-R-Set.

Suit structure:
  Suit name  | Max value | Card count
  -----------+-----------+-----------
  Zeros      |     0     |     1
  Twos       |     2     |     3
  Fours      |     4     |     5
  Sixes      |     6     |     7
  Eights     |     8     |     9
  Tens       |    10     |    11
  Twelves    |    12     |    13
  S/S        |    -1     |     1   (lowest trump, unique)
  -----------+-----------+-----------
  Total                       50

Bonus point cards (added to trick value when won):
  +3 pts : S/S
  +1 pt  : 1/2  (Twos-1),   2/4  (Fours-2),  3/6  (Sixes-3)
  +2 pts : 4/8  (Eights-4), 5/10 (Tens-5),   6/12 (Twelves-6)

Card naming convention: "V/S" where V = value, S = suit's max value.
e.g. "4/8" = value 4 in the suit of Eights.

Each card is represented as a frozen dataclass for safety.
Cards have a unique integer index (0–49) for use in Gym obs/action vectors.
"""

from dataclasses import dataclass
from typing import Optional
import random


# ── Suit definitions ──────────────────────────────────────────────────────────

# (suit_name, max_value)  →  card count = max_value + 1
SUIT_DEFS = [
    ("Zeros",   0),
    ("Twos",    2),
    ("Fours",   4),
    ("Sixes",   6),
    ("Eights",  8),
    ("Tens",   10),
    ("Twelves",12),
]

SS_SUIT  = "S/S"
SS_VALUE = -1

# Bonus points keyed by (suit_name, value).
# The naming pattern V/S means: value V in the suit whose max is S.
BONUS_POINTS: dict[tuple, int] = {
    (SS_SUIT,SS_VALUE): 3,   # S/S wild card
    ("Twos",        1): 1,   # 1/2
    ("Fours",       2): 1,   # 2/4
    ("Sixes",       3): 1,   # 3/6
    ("Eights",      4): 2,   # 4/8
    ("Tens",        5): 2,   # 5/10
    ("Twelves",     6): 2,   # 6/12
}

TOTAL_BONUS_POINTS = 12


# ── Card dataclass ────────────────────────────────────────────────────────────

@dataclass(frozen=True)
class Card:
    """
    Immutable representation of a single Some-R-Set card.

    Attributes
    ----------
    index        : unique integer 0-49; used as the Gym action/obs index
    suit         : suit name string  (e.g. "Eights", "S/S")
    value        : integer rank within the suit
    is_wild      : True means can be laid on any trump suit, i.e. the S/S and 0/0 cards
    bonus_points : extra points added to trick value when this card is won
    """
    index        : int
    suit         : str
    value        : Optional[int]
    is_wild      : bool = False
    bonus_points : int  = 0

    @property
    def short_name(self) -> str:
        """
        Canonical V/S notation, e.g. '4/8', '0/0', 'S/S'.
        For regular cards: '{value}/{suit_max}'.
        """
        if self.suit == SS_SUIT:
            return "S/S"
        suit_max = next(mx for nm, mx in SUIT_DEFS if nm == self.suit)
        return f"{self.value}/{suit_max}"

    def __str__(self):
        bp = f"  [+{self.bonus_points} pts]" if self.bonus_points else ""
        if self.is_wild:
            return f"S/S (wild trump){bp}"
        return f"{self.short_name} ({self.suit}){bp}"

    def __repr__(self):
        return (f"Card(index={self.index}, suit={self.suit!r}, "
                f"value={self.value}, bonus={self.bonus_points})")


# ── Build the full 50-card deck ───────────────────────────────────────────────

def _build_deck() -> list[Card]:
    cards = []
    idx = 0

    for suit_name, max_val in SUIT_DEFS:
        for value in range(max_val + 1):
            bonus = BONUS_POINTS.get((suit_name, value), 0)
            cards.append(Card(
                index        = idx,
                suit         = suit_name,
                value        = value,
                bonus_points = bonus,
            ))
            idx += 1

    # S/S wild card — always index 49
    cards.append(Card(
        index        = idx,
        suit         = SS_SUIT,
        value        = SS_VALUE,
        is_wild      = True,
        bonus_points = BONUS_POINTS[(SS_SUIT, SS_VALUE)],
    ))

    assert len(cards) == 50, f"Expected 50 cards, got {len(cards)}"
    assert sum(c.bonus_points for c in cards) == TOTAL_BONUS_POINTS, \
        "Bonus point total mismatch"
    return cards


# Module-level immutable master deck — treat as read-only
MASTER_DECK: list[Card] = _build_deck()

# Fast lookups
INDEX_TO_CARD:      dict[int,   Card] = {c.index: c for c in MASTER_DECK}
SUIT_VALUE_TO_CARD: dict[tuple, Card] = {(c.suit, c.value): c for c in MASTER_DECK}
BONUS_CARD_INDICES: frozenset[int]    = frozenset(
    c.index for c in MASTER_DECK if c.bonus_points > 0
)


# ── Deck utilities ────────────────────────────────────────────────────────────

def new_shuffled_deck(rng: Optional[random.Random] = None) -> list[Card]:
    """Return a fresh shuffled copy of the 50-card deck."""
    deck = list(MASTER_DECK)
    (rng or random).shuffle(deck)
    return deck


def deal(deck: list[Card], num_players: int) -> list[list[Card]]:
    """
    Deal the entire deck round-robin among num_players.
    Returns a list of hands. With 50 cards and 4 players,
    players 0 and 1 receive 13 cards; players 2 and 3 receive 12.
    """
    hands: list[list[Card]] = [[] for _ in range(num_players)]
    for i, card in enumerate(deck):
        hands[i % num_players].append(card)
    return hands


def trick_bonus(trick: list[Card]) -> int:
    """Sum of bonus points carried by cards in a completed trick."""
    return sum(c.bonus_points for c in trick)


def cards_by_suit(cards: list[Card]) -> dict[str, list[Card]]:
    """Group cards by suit, sorted by value within each suit."""
    groups: dict[str, list[Card]] = {}
    for card in cards:
        groups.setdefault(card.suit, []).append(card)
    for suit in groups:
        groups[suit].sort(key=lambda c: (c.value is None, c.value))
    return groups


def suit_order() -> list[str]:
    """Suits in ascending order of max value, S/S last."""
    return [name for name, _ in SUIT_DEFS] + [SS_SUIT]


# ── Trump helpers ─────────────────────────────────────────────────────────────

def trump_cards(trump_suit: str) -> list[Card]:
    """
    All trump cards in ascending trump order (lowest first).
    S/S is always the lowest trump regardless of suit called.
    """
    ss = INDEX_TO_CARD[49]
    suit_trumps = sorted(
        (c for c in MASTER_DECK if c.suit == trump_suit and not c.is_wild),
        key=lambda c: c.value,
    )
    return [ss] + suit_trumps   # S/S = lowest trump


# ── Pretty-print helpers ──────────────────────────────────────────────────────

def hand_str(hand: list[Card]) -> str:
    """Human-readable hand, grouped by suit, bonus cards flagged."""
    groups = cards_by_suit(hand)
    lines = []
    for suit in suit_order():
        if suit not in groups:
            continue
        parts = []
        for c in groups[suit]:
            label = "wild" if c.is_wild else str(c.value)
            if c.bonus_points:
                label += f"(+{c.bonus_points})"
            parts.append(label)
        lines.append(f"  {suit:8s}: {', '.join(parts)}")
    return "\n".join(lines)


# ── Sanity check / demo ───────────────────────────────────────────────────────

if __name__ == "__main__":
    print(f"Total cards : {len(MASTER_DECK)}")
    print(f"Bonus cards : {len(BONUS_CARD_INDICES)}  "
          f"(total bonus points in deck: {TOTAL_BONUS_POINTS})")
    print()

    print("Full deck — bonus cards highlighted:")
    for suit_name, cards in cards_by_suit(MASTER_DECK).items():
        for c in cards:
            flag = f"  ← +{c.bonus_points} pts" if c.bonus_points else ""
            print(f"  {c.short_name:6s}  {suit_name:8s}  idx={c.index:2d}{flag}")
    print()

    # 4-player deal
    rng   = random.Random(42)
    deck  = new_shuffled_deck(rng)
    hands = deal(deck, 4)
    print("Sample 4-player deal (seed=42):")
    for i, hand in enumerate(hands):
        bp = sum(c.bonus_points for c in hand)
        print(f"\nPlayer {i} ({len(hand)} cards, {bp} bonus pts in hand):")
        print(hand_str(hand))

    # Trick bonus example
    print("\nExample: player wins a trick containing 4/8 and S/S:")
    trick = [
        SUIT_VALUE_TO_CARD[("Eights", 4)],   # +2 pts
        INDEX_TO_CARD[49],                    # S/S +3 pts
        SUIT_VALUE_TO_CARD[("Tens",    3)],   # +0 pts
        SUIT_VALUE_TO_CARD[("Sixes",   2)],   # +0 pts
    ]
    print(f"  Cards : {', '.join(str(c) for c in trick)}")
    print(f"  Bonus : +{trick_bonus(trick)} pts")


# Trick Playing Rules

In [ ]:
"""
Trick-playing and legal-move logic for Some-R-Set.

Key rules encoded here:
  1. Leading:
       - If trump was named, first card of the FIRST trick must be trump.
       - Subsequent tricks (and no-trump hands) may lead any card.

  2. Following (given a led suit):
       - Must follow led suit IF you have one, UNLESS you play:
           a) the 0/0 card  (always legal)
           b) a trump card  (always legal when trump was named)
       - If you have no card of the led suit, play anything.

  3. Trick winner:
       Priority 1 — Any trump card was played  → highest trump value wins.
                    (S/S is the lowest trump, so it loses to any suited trump.)
       Priority 2 — No trump, but 0/0 was played → highest "double" card wins.
                    Doubles are: 0/0, 2/2, 4/4, 6/6, 8/8, 10/10, 12/12.
                    0/0 is the lowest double.
       Priority 3 — No trump, no 0/0 → highest value in the led suit wins.
"""


# ── Constants derived from the deck ──────────────────────────────────────────

# The 0/0 card (Zeros suit, value 0, index 0)
CARD_0_0: Card = SUIT_VALUE_TO_CARD[("Zeros", 0)]

# All "double" cards: the highest-value card of each regular suit
# (the card whose value == suit max, i.e. 0/0, 2/2, 4/4 … 12/12)
# Sorted by value ascending so 0/0 is lowest double.
DOUBLE_CARDS: list[Card] = sorted(
    [SUIT_VALUE_TO_CARD[(suit, max_val)] for suit, max_val in SUIT_DEFS],
    key=lambda c: c.value,
)
DOUBLE_INDICES: frozenset[int] = frozenset(c.index for c in DOUBLE_CARDS)

# S/S wild card
CARD_SS: Card = INDEX_TO_CARD[49]


# ── Legal move logic ──────────────────────────────────────────────────────────

def legal_plays(
    hand:        list[Card],
    led_card:    Card | None,
    trump_suit:  str  | None,
    is_first_trick: bool = False,
    is_leading:  bool = False,
) -> list[Card]:
    """
    Return the subset of `hand` that may legally be played.

    Parameters
    ----------
    hand           : cards held by the current player
    led_card       : the card that opened this trick (None if this player leads)
    trump_suit     : the named trump suit, or None for no-trump
    is_first_trick : True only for the very first trick of the hand
    is_leading     : True when this player is the one leading the trick
    """
    if not hand:
        return []

    # ── Leading ──────────────────────────────────────────────────────
    if is_leading or led_card is None:
        if is_first_trick and trump_suit is not None:
            # Must lead a trump card to open the hand
            trump_cards = [c for c in hand if _is_trump(c, trump_suit)]
            # (Should always be non-empty if rules are followed, but fall back
            # to any card defensively.)
            return trump_cards if trump_cards else hand[:]
        return hand[:]   # any card is fine otherwise

    # ── Following ─────────────────────────────────────────────────────
    led_suit = _effective_suit(led_card, trump_suit)
    same_suit = [c for c in hand if _effective_suit(c, trump_suit) == led_suit]

    if not same_suit:
        # No matching suit — free to play anything
        return hand[:]

    # Has matching suit: must follow, BUT 0/0 and trump are always allowed
    always_legal = []
    if CARD_0_0 in hand:
        always_legal.append(CARD_0_0)
    if trump_suit is not None:
        always_legal += [c for c in hand if _is_trump(c, trump_suit)
                         and c not in always_legal]

    # Combine, deduplicate, preserve order
    legal = list(same_suit)
    for c in always_legal:
        if c not in legal:
            legal.append(c)
    return legal


# ── Trick winner logic ────────────────────────────────────────────────────────

def trick_winner(
    trick:       list[tuple[Card, int]],   # (card, player_index) in play order
    trump_suit:  str | None,
) -> tuple[Card, int]:
    """
    Determine which (card, player) wins the trick.

    Parameters
    ----------
    trick      : list of (card, player_index) in the order they were played.
                 trick[0] is the led card.
    trump_suit : named trump suit, or None.

    Returns
    -------
    (winning_card, winning_player_index)
    """
    if not trick:
        raise ValueError("trick is empty")

    led_card = trick[0][0]

    # ── Priority 1: trump ─────────────────────────────────────────────
    trump_plays = [(c, p) for c, p in trick
                   if trump_suit and _is_trump(c, trump_suit)]
    if trump_plays:
        return max(trump_plays, key=lambda cp: _trump_rank(cp[0], trump_suit))

    # ── Priority 2: 0/0 was led or played → doubles hierarchy ─────────
    played_cards = [c for c, _ in trick]
    if CARD_0_0 in played_cards:
        double_plays = [(c, p) for c, p in trick if c.index in DOUBLE_INDICES]
        if double_plays:
            return max(double_plays, key=lambda cp: cp[0].value)
        # 0/0 was played but no other doubles — 0/0 wins by default
        return next((c, p) for c, p in trick if c == CARD_0_0)

    # ── Priority 3: highest card in led suit ───────────────────────────
    led_suit = led_card.suit
    led_suit_plays = [(c, p) for c, p in trick if c.suit == led_suit]
    return max(led_suit_plays, key=lambda cp: cp[0].value)


# ── Trump-rank helper ─────────────────────────────────────────────────────────

def _trump_rank(card: Card, trump_suit: str) -> int:
    """
    Numeric rank for comparing trump cards.
    S/S = -1 (lowest trump); suited trump cards rank by their face value.
    """
    if card.suit == SS_SUIT:
        return -1          # S/S is always the lowest trump
    return card.value


def _is_trump(card: Card, trump_suit: str | None) -> bool:
    """True if the card belongs to the trump suit or is the S/S wild."""
    if card.suit == SS_SUIT:
        return True
    if trump_suit is None:
        return False
    return card.suit == trump_suit


def _effective_suit(card: Card, trump_suit: str | None) -> str:
    """
    The suit a card 'counts as' for following-suit purposes.
    S/S and trump-suited cards all count as trump_suit when trump is named.
    """
    if _is_trump(card, trump_suit):
        return trump_suit
    return card.suit


# ── Trick summary helper ──────────────────────────────────────────────────────

def trick_summary(
    trick:      list[tuple[Card, int]],
    trump_suit: str | None,
) -> dict:
    """
    Return a dict summarising a completed trick — useful for logging/reward.
    """
    winner_card, winner_player = trick_winner(trick, trump_suit)
    bonus = sum(c.bonus_points for c, _ in trick)
    return {
        "winner_player" : winner_player,
        "winner_card"   : winner_card,
        "bonus_points"  : bonus,
        "cards_played"  : [c for c, _ in trick],
        "trump_suit"    : trump_suit,
    }


# ── Tests / demo ──────────────────────────────────────────────────────────────

if __name__ == "__main__":

    def card(suit, val):
        return SUIT_VALUE_TO_CARD[(suit, val)]

    def ss():
        return CARD_SS

    def show_trick(label, trick, trump):
        s = trick_summary(trick, trump)
        print(f"\n{label}")
        print(f"  Trump : {trump or 'none'}")
        for c, p in trick:
            print(f"    Player {p}: {c}")
        print(f"  → Winner: Player {s['winner_player']} "
              f"with {s['winner_card']}  (bonus: +{s['bonus_points']} pts)")

    # 1. Trump wins over led suit
    show_trick(
        "1. Trump card beats led suit",
        trick=[
            (card("Eights", 8), 0),   # leads 8/8 (highest Eights)
            (card("Eights", 3), 1),   # follows suit
            (card("Tens",   2), 2),   # plays trump (Tens is trump)
            (card("Eights", 5), 3),
        ],
        trump="Tens",
    )

    # 2. S/S (lowest trump) still beats all non-trump
    show_trick(
        "2. S/S beats non-trump even though it's lowest trump",
        trick=[
            (card("Twelves", 12), 0),  # leads highest Twelves
            (card("Twelves",  9), 1),
            (ss(),                2),  # plays S/S (trump = Sixes)
            (card("Twelves",  7), 3),
        ],
        trump="Sixes",
    )

    # 3. 0/0 triggers doubles: highest double wins
    show_trick(
        "3. 0/0 played — doubles hierarchy takes over",
        trick=[
            (card("Sixes",  6), 0),   # leads 6/6 (a double!)
            (card("Sixes",  4), 1),
            (CARD_0_0,          2),   # plays 0/0 — activates doubles rule
            (card("Tens",  10), 3),   # 10/10 is the highest double here
        ],
        trump=None,
    )

    # 4. 0/0 played but no other doubles — 0/0 itself wins
    show_trick(
        "4. 0/0 played, no other doubles — 0/0 wins",
        trick=[
            (card("Sixes",  5), 0),
            (card("Sixes",  4), 1),
            (CARD_0_0,          2),
            (card("Sixes",  3), 3),
        ],
        trump=None,
    )

    # 5. No trump, no 0/0 — highest in led suit wins
    show_trick(
        "5. Plain trick — highest in led suit wins",
        trick=[
            (card("Fours",  1), 0),
            (card("Fours",  4), 1),   # highest Fours
            (card("Sixes",  6), 2),   # different suit, irrelevant
            (card("Fours",  2), 3),
        ],
        trump=None,
    )

    # 6. Legal moves — must follow suit but may play 0/0 or trump instead
    print("\n6. Legal plays — hand has Eights and trump (Tens), led suit is Eights")
    hand = [
        card("Eights", 3),
        card("Eights", 7),
        card("Tens",   4),   # trump
        card("Sixes",  5),
        CARD_0_0,            # always legal
    ]
    led = card("Eights", 1)
    legal = legal_plays(hand, led_card=led, trump_suit="Tens")
    print(f"  Trump    : Tens")
    print(f"  Led card : {led}")
    print(f"  Hand     : {[str(c) for c in hand]}")
    print(f"  Legal    : {[str(c) for c in legal]}")


# Scoring Rules

In [ ]:
"""
Hand and game scoring logic for Some-R-Set.

Scoring summary:
  - 12 tricks × 1 pt each   = 12 pts
  - Bonus points in deck    = 12 pts  (3+1+1+1+2+2+2)
  - Total per hand          = 24 pts

Teams:
  Team 0: players 0 and 2
  Team 1: players 1 and 3

Bid outcome:
  - Bidding team scores >= bid  → bidding team gets their points; other team gets 0
  - Bidding team scores < bid   → bidding team loses 2 × (bid - scored);
                                   other team gets all their own points

Win condition: first team to reach 66+ points wins the game.
"""

from dataclasses import dataclass, field
from typing import Optional

# ── Constants ─────────────────────────────────────────────────────────────────

NUM_PLAYERS   = 4
NUM_TEAMS     = 2
WINNING_SCORE = 66
MINIMUM_SCORE = -99.  # If a score goes this low, the game ends.

# Team membership: PLAYER_TEAM[player_index] → team index
PLAYER_TEAM = {0: 0, 1: 1, 2: 0, 3: 1}
TEAM_PLAYERS = {0: [0, 2], 1: [1, 3]}

# Total points available each hand
# 13 tricks (one per card per player in a 4-player deal of 50-card deck
# isn't exactly 13 — let's compute properly)
# 50 cards / 4 players = 12.5, so players get 13/13/12/12 cards.
# Number of tricks = cards in shortest hand = 12.
# But extra cards (players 0 & 1 have 13) mean one player runs out last.
# Actually in trick-taking: tricks played = min hand size ... let's just
# track empirically and assert total == 24 as stated in the rules.
POINTS_PER_TRICK  = 1
TOTAL_HAND_POINTS = 24          # as specified in the rules
PENALTY_MULTIPLIER = 2          # missing bid costs 2× the shortfall


# ── Per-hand result dataclass ─────────────────────────────────────────────────

@dataclass
class TrickResult:
    """Record of a single completed trick."""
    trick_number   : int
    winning_player : int
    winning_team   : int
    cards_played   : list          # list[Card]
    bonus_points   : int


@dataclass
class HandResult:
    """
    Full scoring result for one hand.
    Produced by score_hand(); consumed by apply_hand_result().
    """
    # Inputs
    trick_results   : list[TrickResult]
    bid_value       : int
    bid_winner      : int          # player index who won the bid
    bid_team        : int          # team index of bid winner

    # Computed totals
    tricks_won      : dict = field(default_factory=dict)   # player → count
    team_tricks     : dict = field(default_factory=dict)   # team   → count
    team_raw_points : dict = field(default_factory=dict)   # team   → pts before bid adj
    team_delta      : dict = field(default_factory=dict)   # team   → score change
    bid_made        : bool = False
    shortfall       : int  = 0
    total_points_counted: int = 0

    def summary(self) -> str:
        lines = [
            f"  Bid: {self.bid_value} by Player {self.bid_winner} "
            f"(Team {self.bid_team})",
            f"  Bid made: {self.bid_made}  "
            f"(shortfall: {self.shortfall})",
            f"  Total points counted: {self.total_points_counted}",
            f"  Team raw points: {dict(self.team_raw_points)}",
            f"  Score deltas:    {dict(self.team_delta)}",
        ]
        return "\n".join(lines)


# ── Core scoring function ─────────────────────────────────────────────────────

def score_hand(
    trick_results : list[TrickResult],
    bid_value     : int,
    bid_winner    : int,
) -> HandResult:
    """
    Compute the score delta for each team after a completed hand.

    Parameters
    ----------
    trick_results : list of TrickResult, one per trick played
    bid_value     : the winning bid amount
    bid_winner    : player index who won the bid

    Returns
    -------
    HandResult with all fields populated
    """
    bid_team = PLAYER_TEAM[bid_winner]

    # ── Tally tricks and points per team ─────────────────────────────
    tricks_won      = {p: 0 for p in range(NUM_PLAYERS)}
    team_tricks     = {t: 0 for t in range(NUM_TEAMS)}
    team_raw_points = {t: 0 for t in range(NUM_TEAMS)}

    for tr in trick_results:
        p = tr.winning_player
        t = tr.winning_team
        tricks_won[p]       += 1
        team_tricks[t]      += 1
        team_raw_points[t]  += POINTS_PER_TRICK + tr.bonus_points

    total = sum(team_raw_points.values())

    # Sanity check — warn but don't crash (rules say 24, verify empirically)
    if total != TOTAL_HAND_POINTS:
        import warnings
        warnings.warn(
            f"Expected {TOTAL_HAND_POINTS} total hand points, got {total}. "
            "Check trick bonus tallying or number of tricks played."
        )

    # ── Apply bid outcome ─────────────────────────────────────────────
    bid_team_points = team_raw_points[bid_team]
    other_team      = 1 - bid_team
    other_points    = team_raw_points[other_team]

    bid_made  = bid_team_points >= bid_value
    shortfall = max(0, bid_value - bid_team_points)

    team_delta = {}
    if bid_made:
        # Bidding team earns their points; other team earns nothing this hand
        team_delta[bid_team]  =  bid_team_points
        team_delta[other_team] = 0
    else:
        # Bidding team penalised; other team keeps their raw score
        team_delta[bid_team]   = -(shortfall * PENALTY_MULTIPLIER)
        team_delta[other_team] =  other_points

    result = HandResult(
        trick_results        = trick_results,
        bid_value            = bid_value,
        bid_winner           = bid_winner,
        bid_team             = bid_team,
        tricks_won           = tricks_won,
        team_tricks          = team_tricks,
        team_raw_points      = team_raw_points,
        team_delta           = team_delta,
        bid_made             = bid_made,
        shortfall            = shortfall,
        total_points_counted = total,
    )
    return result


# ── Game state ────────────────────────────────────────────────────────────────

@dataclass
class GameState:
    """
    Tracks cumulative scores across hands and whose turn it is to bid first.

    Attributes
    ----------
    team_scores      : running total for each team (can go negative)
    hand_number      : how many hands have been played
    first_bidder     : player index who bids first this hand
                       (rotates by 1 each hand)
    hand_history     : list of HandResult objects, one per completed hand
    winning_team     : None until the game ends
    """
    team_scores  : dict = field(default_factory=lambda: {0: 0, 1: 0})
    hand_number  : int  = 0
    first_bidder : int  = 0
    hand_history : list = field(default_factory=list)
    winning_team : Optional[int] = None

    def apply_hand_result(self, result: HandResult) -> None:
        """Update game state after a completed hand."""
        for team, delta in result.team_delta.items():
            self.team_scores[team] += delta

        self.hand_history.append(result)
        self.hand_number  += 1
        self.first_bidder  = (self.first_bidder + 1) % NUM_PLAYERS

        # Check win condition
        for team, score in self.team_scores.items():
            if score >= WINNING_SCORE:
                # If both teams hit winning score on the same hand, higher score wins
                if self.winning_team is None:
                    self.winning_team = team
                else:
                    # Already found one — pick the higher scorer
                    other = 1 - team
                    if self.team_scores[team] > self.team_scores[other]:
                        self.winning_team = team
            elif score <= MINIMUM_SCORE:
                # If both teams hit minimum score on the same hand, higher score wins
                if self.winning_team is None:
                    self.winning_team = (team + 1) % 2
                else:
                    # Already found one — pick the higher scorer
                    other = 1 - team
                    if self.team_scores[team] > self.team_scores[other]:
                        self.winning_team = team


    @property
    def game_over(self) -> bool:
        return self.winning_team is not None

    def score_str(self) -> str:
        return (f"Hand {self.hand_number} | "
                f"Team 0 (P0,P2): {self.team_scores[0]:+d}  "
                f"Team 1 (P1,P3): {self.team_scores[1]:+d}  "
                f"Next first bidder: P{self.first_bidder}")


# ── Helper: build TrickResult from raw trick data ─────────────────────────────

def make_trick_result(
    trick_number   : int,
    winning_player : int,
    cards_played   : list,        # list[Card]
) -> TrickResult:
    """Convenience constructor — computes bonus and team automatically."""
    bonus = sum(c.bonus_points for c in cards_played)
    return TrickResult(
        trick_number   = trick_number,
        winning_player = winning_player,
        winning_team   = PLAYER_TEAM[winning_player],
        cards_played   = cards_played,
        bonus_points   = bonus,
    )


# ── Demo / tests ──────────────────────────────────────────────────────────────

if __name__ == "__main__":
    def card(suit, val):  return SUIT_VALUE_TO_CARD[(suit, val)]
    def ss():             return INDEX_TO_CARD[49]

    game = GameState()
    print(f"Starting game.  Win target: {WINNING_SCORE} pts\n")

    # ── Hand 1: bid made ──────────────────────────────────────────────
    # Team 0 bids 10, scores 18 (bid made)
    # We'll fake 12 tricks with carefully chosen bonus cards
    tricks_h1 = []
    # Team 0 wins tricks 0-7 (8 tricks), Team 1 wins tricks 8-11 (4 tricks)
    # Bonus cards: 6/12, 4/8(+2) and S/S(+3) go to Team 0;
    #            : 1/2, 2/4, 3/6(+1) and 5/10(+2) to Team 1
    # Team 0 raw = 8 tricks + 2 + 2 + 3 = 15 pts
    # Team 1 raw = 4 tricks + 1 + 1 + 1 +2 =  9 pts   → total 24 ✓

    bonus_to_t0 = [
        [card("Eights", 4)],           # +2 pts  → trick 2
        [ss()],                        # +3 pts  → trick 5
        [card("Twelves", 6)],          # +2 pts  → trick 6
    ]
    bonus_to_t1 = [
        [card("Twos",   1)],           # +1 pt   → trick 8
        [card("Fours",  2)],           # +1 pt   → trick 9
        [card("Tens",   5)],           # +2 pts  → trick 10
        [card("Sixes",  3)],           # +1 pt   → trick 11
    ]

    for i in range(12):
        if i < 8:
            extra = bonus_to_t0.pop(0) if bonus_to_t0 and i in [2,5,6] else []
            winner = 0 if i % 2 == 0 else 2   # alternates between team 0 players
            played = [card("Twelves", i % 6)] + extra
        else:
            extra = bonus_to_t1.pop(0) if bonus_to_t1 else []
            winner = 1 if i % 2 == 0 else 3   # alternates between team 1 players
            played = [card("Tens", i % 5)] + extra
        tricks_h1.append(make_trick_result(i, winner, played))

    result1 = score_hand(tricks_h1, bid_value=10, bid_winner=0)
    game.apply_hand_result(result1)
    print("── Hand 1: Team 0 bids 10 ──────────────────────────────────")
    print(result1.summary())
    print(f"  {game.score_str()}\n")

    # ── Hand 2: bid missed ────────────────────────────────────────────
    # Team 1 bids 16 but scores only 10 → shortfall 6 → penalty -12
    # Team 0 scores 14
    tricks_h2 = []
    bonus_to_t0 = [
        [card("Eights", 4)],           # +2 pts  → trick 2
        [ss()],                        # +3 pts  → trick 5
    ]
    bonus_to_t1 = [
        [card("Twos",   1)],           # +1 pt   → trick 8
        [card("Fours",  2)],           # +1 pt   → trick 9
        [card("Twelves", 6)],          # +2 pts  → trick 6
        [card("Tens",   5)],           # +2 pts  → trick 10
        [card("Sixes",  3)],           # +1 pt   → trick 11
    ]
    for i in range(12):
        if i < 5:   # Team 0 wins first 5
            winner = 0
            extra = bonus_to_t0.pop(0) if bonus_to_t0 else []
            played = [card("Twelves", i % 6)] + extra
        else:        # Team 1 wins last 7
            winner = 1
            extra = bonus_to_t1.pop(0) if bonus_to_t1 else []
            played = [card("Tens", i % 5)] + extra
        tricks_h2.append(make_trick_result(i, winner, played))

    result2 = score_hand(tricks_h2, bid_value=16, bid_winner=1)
    game.apply_hand_result(result2)
    print("── Hand 2: Team 1 bids 16, misses ──────────────────────────")
    print(result2.summary())
    print(f"  {game.score_str()}\n")

    # ── Simulate hands until someone wins ────────────────────────────
    print("── Simulating remaining hands (Team 0 bids 12, makes it) ───!!!!")
    import random
    rng = random.Random(7)
    hand = 3
    while not game.game_over:
        # fake hand: Team 0 always wins 8 tricks + 7 bonus pts
        bonus_to_t0 = [
            [card("Twos",   1)],           # +1 pt
            [card("Fours",  2)],           # +1 pt
            [card("Twelves", 6)],          # +2 pts
            [card("Tens",   5)],           # +2 pts
            [card("Sixes",  3)],           # +1 pt
        ]
        bonus_to_t1 = [
            [card("Eights", 4)],           # +2 pts
            [ss()],                        # +3 pts
        ]
        tr = []
        for i in range(12):
            winner = 0 if i < 8 else 1
            played = [card("Twelves", 0)]
            if winner == 0:
                played += bonus_to_t0.pop(0) if bonus_to_t0 else []
            else:
                played += bonus_to_t1.pop(0) if bonus_to_t1 else []
            tr.append(make_trick_result(i, winner, played))
        res = score_hand(tr, bid_value=10, bid_winner=0)
        game.apply_hand_result(res)
        print(f"  After hand {hand}: {game.score_str()}")
        hand += 1

    print(f"\n🏆  Game over — Team {game.winning_team} wins!")
    print(f"  Final scores: Team 0 = {game.team_scores[0]}, "
          f"Team 1 = {game.team_scores[1]}")
    print(f"  Total hands played: {game.hand_number}")


# Bidding

In [ ]:
"""
Dealing (with kitty) and bidding logic for Some-R-Set.

Dealing:
  - 50 cards dealt 12 to each of 4 players; 2 remainder form the kitty.
  - Kitty is face-up and public; both cards go into the first trick.
  - Whoever wins the first trick collects the kitty cards (and their bonuses).

Bidding:
  - Starts with the player to the left of the dealer (first_bidder).
  - Each player may bid or pass; bids must strictly exceed the current high bid.
  - Minimum bid: MIN_BID (default 6). Enforced to prevent misdeal Nash traps.
  - Bidding ends when three consecutive players pass after a bid is made,
    OR (rarely in practice) all four players pass → misdeal.
  - The bid winner names trump (or declares no-trump) and leads the first trick.
  - Bid winner leads the first trick, so the kitty bonus points are theirs
    to win (or lose if carless).
"""

from dataclasses import dataclass, field
from typing import Optional
import random

# ── Constants ─────────────────────────────────────────────────────────────────

NUM_PLAYERS   = 4
CARDS_PER_HAND = 12
KITTY_SIZE    = 2          # 50 - 4×12 = 2
MIN_BID       = 6          # enforced floor; prevents misdeal equilibrium
MAX_BID       = 24         # total points in a hand
PASS          = -1         # sentinel value for "pass" action

assert CARDS_PER_HAND * NUM_PLAYERS + KITTY_SIZE == 50


# ── Deal ──────────────────────────────────────────────────────────────────────

@dataclass
class Deal:
    """Result of a single deal."""
    hands        : list[list[Card]]   # hands[player_index] → list of Card
    kitty        : list[Card]         # 2 face-up cards, part of first trick
    kitty_bonus  : int                # total bonus pts carried by kitty cards
    dealer       : int                # player who dealt (bids last)
    first_bidder : int                # player who bids first (left of dealer)

    def kitty_str(self) -> str:
        parts = []
        for c in self.kitty:
            label = c.short_name
            if c.bonus_points:
                label += f"(+{c.bonus_points})"
            parts.append(label)
        return ", ".join(parts)


def new_deal(dealer: int, rng: Optional[random.Random] = None) -> Deal:
    """
    Shuffle and deal 12 cards to each player; remaining 2 form the kitty.
    Dealer index determines first_bidder = (dealer + 1) % NUM_PLAYERS.
    """
    deck = new_shuffled_deck(rng)
    hands = [deck[i * CARDS_PER_HAND:(i + 1) * CARDS_PER_HAND]
             for i in range(NUM_PLAYERS)]
    kitty = deck[NUM_PLAYERS * CARDS_PER_HAND:]

    assert len(kitty) == KITTY_SIZE
    assert all(len(h) == CARDS_PER_HAND for h in hands)

    kitty_bonus  = sum(c.bonus_points for c in kitty)
    first_bidder = (dealer + 1) % NUM_PLAYERS

    return Deal(
        hands        = hands,
        kitty        = kitty,
        kitty_bonus  = kitty_bonus,
        dealer       = dealer,
        first_bidder = first_bidder,
    )


# ── Bidding state ─────────────────────────────────────────────────────────────

@dataclass
class BiddingState:
    """
    Tracks the live bidding round.

    The bidding sequence visits players in order starting from first_bidder.
    A player's turn produces either a bid (integer >= MIN_BID and > current_bid)
    or a pass (PASS sentinel).

    Bidding ends when:
      a) Three consecutive passes follow the current high bid  → bid_winner set.
      b) All four players pass on the first pass-around       → misdeal.
    """
    first_bidder   : int
    current_bid    : int  = 0           # 0 = no bid yet
    current_bidder : int  = -1          # player who holds the current bid
    bid_history    : list = field(default_factory=list)  # (player, bid|PASS)
    consecutive_passes: int = 0
    bids_placed    : int  = 0           # how many non-pass bids so far
    is_complete    : bool = False
    is_misdeal     : bool = False
    bid_winner     : Optional[int] = None
    winning_bid    : Optional[int] = None

    # Whose turn it is to bid right now
    _turn_index    : int = 0            # index into rotation order

    @property
    def rotation(self) -> list[int]:
        """Players in bid order starting from first_bidder."""
        return [(self.first_bidder + i) % NUM_PLAYERS
                for i in range(NUM_PLAYERS)]

    @property
    def current_player(self) -> int:
        return self.rotation[self._turn_index % NUM_PLAYERS]

    def legal_bids(self) -> list[int]:
        """
        List of legal bid values for the current player.
        Includes PASS (always legal).
        Returns empty list if bidding is already complete.
        """
        if self.is_complete:
            return []
        floor = max(MIN_BID, self.current_bid + 1)
        return [PASS] + list(range(floor, MAX_BID + 1))

    def can_bid(self, amount: int) -> bool:
        return amount == PASS or (
            amount >= MIN_BID
            and amount > self.current_bid
            and amount <= MAX_BID
        )

    def place_bid(self, player: int, amount: int) -> None:
        """
        Record a bid or pass from `player`.
        Raises ValueError on illegal action or wrong player.
        """
        if self.is_complete:
            raise ValueError("Bidding is already complete.")
        if player != self.current_player:
            raise ValueError(
                f"It is Player {self.current_player}'s turn, not Player {player}.")
        if not self.can_bid(amount):
            raise ValueError(
                f"Bid {amount} is illegal. Must be PASS or "
                f">= {max(MIN_BID, self.current_bid + 1)}.")

        self.bid_history.append((player, amount))

        if amount == PASS:
            self.consecutive_passes += 1
        else:
            self.current_bid        = amount
            self.current_bidder     = player
            self.consecutive_passes = 0
            self.bids_placed       += 1

        self._turn_index += 1
        self._check_completion()

    def _check_completion(self) -> None:
        # All four passed without a bid → misdeal
        if (self.bids_placed == 0
                and self.consecutive_passes == NUM_PLAYERS):
            self.is_complete = True
            self.is_misdeal  = True
            return

        # Three consecutive passes after a bid → bidding over
        if self.bids_placed > 0 and self.consecutive_passes == NUM_PLAYERS - 1:
            self.is_complete  = True
            self.bid_winner   = self.current_bidder
            self.winning_bid  = self.current_bid

    def summary(self) -> str:
        lines = [f"Bidding (first bidder: P{self.first_bidder})"]
        for player, amount in self.bid_history:
            label = "PASS" if amount == PASS else str(amount)
            lines.append(f"  P{player}: {label}")
        if self.is_misdeal:
            lines.append("  → MISDEAL (all passed)")
        elif self.is_complete:
            lines.append(
                f"  → P{self.bid_winner} wins bid of {self.winning_bid}")
        else:
            lines.append(f"  → Waiting for P{self.current_player} "
                         f"(current bid: {self.current_bid or 'none'})")
        return "\n".join(lines)


def new_bidding(first_bidder: int) -> BiddingState:
    return BiddingState(first_bidder=first_bidder, _turn_index=0)


# ── Trump declaration (follows bid) ──────────────────────────────────────────

@dataclass
class TrumpDeclaration:
    """The bid winner's trump choice after winning the bid."""
    bid_winner  : int
    trump_suit  : Optional[str]    # None = no-trump declared
    no_trump    : bool             # True if no-trump chosen

    def __str__(self):
        if self.no_trump:
            return f"Player {self.bid_winner} declares NO TRUMP"
        return f"Player {self.bid_winner} declares trump: {self.trump_suit}"


VALID_TRUMP_SUITS = ["Zeros", "Twos", "Fours", "Sixes",
                     "Eights", "Tens", "Twelves"]

def declare_trump(bid_winner: int,
                  trump_suit: Optional[str]) -> TrumpDeclaration:
    """Validate and record the trump declaration."""
    if trump_suit is not None and trump_suit not in VALID_TRUMP_SUITS:
        raise ValueError(f"'{trump_suit}' is not a valid trump suit.")
    return TrumpDeclaration(
        bid_winner = bid_winner,
        trump_suit = trump_suit,
        no_trump   = trump_suit is None,
    )


# ── Observation vector helpers ────────────────────────────────────────────────

def bidding_observation(
    player       : int,
    hand         : list[Card],
    kitty        : list[Card],
    bidding      : BiddingState,
) -> dict:
    """
    Return a dict of everything the current bidder can observe.
    This will later be flattened into the Gym observation vector.

    Visible information during bidding:
      - Own 12-card hand
      - Both kitty cards (face-up)
      - Current high bid
      - Which players have passed / bid (but not their hands)
    """
    pass_flags = [False] * NUM_PLAYERS
    bid_amounts = [0] * NUM_PLAYERS
    for p, amt in bidding.bid_history:
        if amt == PASS:
            pass_flags[p] = True
        else:
            bid_amounts[p] = amt

    return {
        "player"        : player,
        "hand"          : hand,
        "kitty"         : kitty,
        "kitty_bonus"   : sum(c.bonus_points for c in kitty),
        "current_bid"   : bidding.current_bid,
        "current_bidder": bidding.current_bidder,
        "pass_flags"    : pass_flags,
        "bid_amounts"   : bid_amounts,
        "legal_bids"    : bidding.legal_bids(),
    }


# ── Demo ──────────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    rng    = random.Random(42)
    dealer = 0
    deal   = new_deal(dealer, rng)

    print("═" * 55)
    print(f"New deal  (dealer: P{deal.dealer}, "
          f"first bidder: P{deal.first_bidder})")
    print(f"Kitty: {deal.kitty_str()}  "
          f"(+{deal.kitty_bonus} bonus pts)")
    for i, hand in enumerate(deal.hands):
        hand_bonus = sum(c.bonus_points for c in hand)
        print(f"\nPlayer {i} hand ({len(hand)} cards, "
              f"+{hand_bonus} bonus pts):")
        print(hand_str(hand))

    # ── Scenario 1: normal competitive bidding ────────────────────────
    print("\n" + "═" * 55)
    print("Scenario 1: competitive bidding")
    b = new_bidding(deal.first_bidder)
    sequence = [
        (1, 8),
        (2, PASS),
        (3, 14),
        (0, 16),
        (1, PASS),
        (2, PASS),
        (3, PASS),
    ]
    for player, amount in sequence:
        b.place_bid(player, amount)
        if b.is_complete:
            break
    print(b.summary())
    if b.bid_winner is not None:
        td = declare_trump(b.bid_winner, "Eights")
        print(td)

    # ── Scenario 2: misdeal (all pass) ───────────────────────────────
    print("\n" + "═" * 55)
    print("Scenario 2: misdeal — all players pass")
    b2 = new_bidding(deal.first_bidder)
    for player in b2.rotation:
        b2.place_bid(player, PASS)
        if b2.is_complete:
            break
    print(b2.summary())

    # ── Scenario 3: no-trump declaration ─────────────────────────────
    print("\n" + "═" * 55)
    print("Scenario 3: no-trump declared")
    b3 = new_bidding(deal.first_bidder)
    b3.place_bid(1, 10)
    b3.place_bid(2, PASS)
    b3.place_bid(3, PASS)
    b3.place_bid(0, PASS)
    print(b3.summary())
    if b3.bid_winner is not None:
        td3 = declare_trump(b3.bid_winner, None)
        print(td3)

    # ── MIN_BID enforcement ───────────────────────────────────────────
    print("\n" + "═" * 55)
    print(f"Minimum bid enforcement (MIN_BID = {MIN_BID})")
    b4 = new_bidding(0)
    try:
        b4.place_bid(0, 3)   # below minimum
    except ValueError as e:
        print(f"  Correctly rejected bid of 3: {e}")
    b4.place_bid(0, MIN_BID)
    print(f"  Bid of {MIN_BID} accepted. Legal bids now start at {MIN_BID+1}.")


# Gymnasium Environment Setup

In [ ]:
"""
Full Gymnasium environment for Some-R-Set.

Phase flow each episode:
  DEALING      → new_deal()         : shuffle, deal 12 cards each + 2 kitty
  BIDDING      → BiddingState       : players bid or pass in rotation
  DECLARING    → TrumpDeclaration   : bid winner names trump (or no-trump)
  PLAYING      → trick loop x 12    : players play cards trick by trick
  SCORING      → score_hand()       : tally points, apply bid outcome
  (repeat from DEALING until a team reaches 66 pts)

Single-agent design:
  The environment manages all four players, but exposes one agent seat
  (default: player 0) as the "learning agent". The other three players
  are controlled by a pluggable OpponentPolicy (random by default).
  All observations are from the perspective of the learning agent.

Observation vector layout  (float32, all values in [0, 1]):
  Segment                        Size   Notes
  ─────────────────────────────────────────────────────────────
  My hand (card present)           50   one-hot per card index
  Kitty cards                      50   one-hot; face-up all game
  Cards seen (played in prior tricks) 50
  Cards in current trick           50   includes kitty in trick 1
  Trump suit (one-hot)              8   7 suits + no-trump flag
  Current high bid (normalised)     1   bid / MAX_BID
  Player pass flags                 4   1 if player has passed
  Scores team 0 & 1 (normalised)   2   score / WINNING_SCORE
  Whose turn (one-hot)              4
  Phase (one-hot)                   4   BIDDING/DECLARING/PLAYING/SCORING
  ─────────────────────────────────────────────────────────────
  Total                           223

Action space:
  During BIDDING   : Discrete(20)  → 0=PASS, 1=bid6, 2=bid7, … 19=bid24
  During DECLARING : Discrete(8)   → 0..6=suit index, 7=no-trump
  During PLAYING   : Discrete(50)  → card index 0-49

  We use a single Discrete(50) space throughout (largest), and encode
  bidding/declaring actions into that same range.  Legal masks are
  always provided in info["legal_actions"].
"""

import gymnasium as gym
from gymnasium import spaces
import numpy as np
import random
from typing import Optional, Callable


# ── Phase constants ────────────────────────────────────────────────────────────

PHASE_BIDDING   = 0
PHASE_DECLARING = 1
PHASE_PLAYING   = 2
PHASE_SCORING   = 3   # transient; env auto-advances after scoring

# ── Action encoding ────────────────────────────────────────────────────────────
# All phases use a Discrete(50) action space.
# Bidding actions are encoded as:  0=PASS, 1=bid MIN_BID, 2=bid MIN_BID+1, …
# Declaring actions:               0..6 = suit index in VALID_TRUMP_SUITS, 7=no-trump
# Playing actions:                 card index 0-49  (direct)

BID_ACTION_PASS   = 0
BID_ACTION_OFFSET = MIN_BID - 1   # action k → bid value k + BID_ACTION_OFFSET

def bid_to_action(bid: int) -> int:
    if bid == PASS: return BID_ACTION_PASS
    return bid - BID_ACTION_OFFSET        # bid 6 → action 1

def action_to_bid(action: int) -> int:
    if action == BID_ACTION_PASS: return PASS
    return action + BID_ACTION_OFFSET     # action 1 → bid 6

DECLARE_NO_TRUMP_ACTION = len(VALID_TRUMP_SUITS)   # = 7

def suit_to_action(suit: Optional[str]) -> int:
    if suit is None: return DECLARE_NO_TRUMP_ACTION
    return VALID_TRUMP_SUITS.index(suit)

def action_to_suit(action: int) -> Optional[str]:
    if action == DECLARE_NO_TRUMP_ACTION: return None
    return VALID_TRUMP_SUITS[action]

# ── Observation layout ─────────────────────────────────────────────────────────

NUM_CARDS       = 50
NUM_SUITS       = len(VALID_TRUMP_SUITS) + 1   # 7 + no-trump flag = 8

OBS_HAND_START       = 0
OBS_KITTY_START      = 50
OBS_SEEN_START       = 100
OBS_TRICK_START      = 150
OBS_TRUMP_START      = 200
OBS_HIGH_BID         = 208
OBS_PASS_FLAGS       = 209
OBS_SCORES           = 213
OBS_WHOSE_TURN       = 215
OBS_PHASE            = 219
OBS_TOTAL            = 223


# ── Default opponent policy: random legal action ───────────────────────────────

def random_opponent_policy(legal_actions: list[int], **kwargs) -> int:
    return random.choice(legal_actions)


# ── Main environment class ─────────────────────────────────────────────────────

class SomeRSetEnv(gym.Env):
    """
    Gymnasium environment for Some-R-Set (single learning agent vs 3 opponents).

    Parameters
    ----------
    agent_player      : which seat (0-3) is the learning agent
    opponent_policy   : callable(legal_actions, obs, phase) → action int
                        defaults to random_opponent_policy
    render_mode       : 'human' for text output, None for silent
    """

    metadata = {"render_modes": ["human"]}

    def __init__(
        self,
        agent_player    : int = 0,
        opponent_policy : Callable = random_opponent_policy,
        render_mode     : Optional[str] = None,
    ):
        super().__init__()
        self.agent_player    = agent_player
        self.opponent_policy = opponent_policy
        self.render_mode     = render_mode

        # Single Discrete(50) covers all phases
        self.action_space = spaces.Discrete(NUM_CARDS)

        # Most dims are 0/1 flags, but score dims (OBS_SCORES) can be in [-1,1]
        obs_low  = np.zeros(OBS_TOTAL, dtype=np.float32)
        obs_high = np.ones(OBS_TOTAL,  dtype=np.float32)
        obs_low[OBS_SCORES : OBS_SCORES + 2]  = -1.0
        self.observation_space = spaces.Box(
            low=obs_low, high=obs_high, dtype=np.float32)

        # Runtime state (populated in reset)
        self.game : Optional[GameState]       = None
        self.deal : Optional[Deal]            = None
        self.bid  : Optional[BiddingState]    = None
        self.trump: Optional[TrumpDeclaration]= None

        self.phase          : int  = PHASE_BIDDING
        self.current_player : int  = 0
        self.trick_number   : int  = 0
        self.current_trick  : list = []   # list of (Card, player_index)
        self.trick_results  : list = []   # TrickResult per completed trick
        self.cards_seen     : set  = set()# card indices played in prior tricks
        self.dealer         : int  = 0

    # ═══════════════════════════════════════════════════════════════════
    # reset()
    # ═══════════════════════════════════════════════════════════════════

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        py_rng = random.Random(seed)

        if self.game is None:
            self.game   = GameState()
            self.dealer = 0
        else:
            # Dealer rotates each hand (first_bidder rotation already in
            # GameState, but dealer is one seat earlier)
            self.dealer = (self.dealer + 1) % NUM_PLAYERS

        self._start_new_hand(py_rng)

        # Let opponents act until it's the agent's turn
        obs, info = self._advance_to_agent()
        return obs, info

    # ═══════════════════════════════════════════════════════════════════
    # step()
    # ═══════════════════════════════════════════════════════════════════

    def step(self, action: int):
        assert not self.game.game_over, "Game is over; call reset()."

        legal = self._legal_actions()
        if action not in legal:
            # Penalise illegal action and return current state unchanged
            obs  = self._get_observation()
            info = {"legal_actions": legal, "illegal_action": True}
            return obs, -1.0, False, False, info

        self._apply_action(self.agent_player, action)

        # Let opponents respond until it's the agent's turn again
        obs, info = self._advance_to_agent()

        reward     = info.pop("reward", 0.0)
        terminated = self.game.game_over
        truncated  = False

        if self.render_mode == "human":
            self.render()

        return obs, reward, terminated, truncated, info

    # ═══════════════════════════════════════════════════════════════════
    # render()
    # ═══════════════════════════════════════════════════════════════════

    def render(self):
        if self.render_mode != "human":
            return
        g = self.game
        print(f"\n{'─'*55}")
        phase_names = ["BIDDING","DECLARING","PLAYING","SCORING"]
        print(f"Hand {g.hand_number+1} | Phase: {phase_names[self.phase]} | "
              f"Trick: {self.trick_number+1}/12")
        print(f"Scores → Team 0: {g.team_scores[0]:+d}  "
              f"Team 1: {g.team_scores[1]:+d}")
        if self.trump:
            ts = self.trump.trump_suit or "NO TRUMP"
            print(f"Trump: {ts}  |  Bid: {self.bid.winning_bid} "
                  f"by P{self.bid.bid_winner}")
        print(f"Kitty: {self.deal.kitty_str()}")
        print(f"Agent (P{self.agent_player}) hand:")
        print(hand_str(self.deal.hands[self.agent_player]))
        if self.current_trick:
            played = ", ".join(
                f"P{p}:{c.short_name}" for c, p in self.current_trick
            )
            print(f"Cards played in current trick: {played}")

    # ═══════════════════════════════════════════════════════════════════
    # Internal: hand setup
    # ═══════════════════════════════════════════════════════════════════

    def _start_new_hand(self, rng: random.Random):
        self.deal         = new_deal(self.dealer, rng)
        self.bid          = new_bidding(self.deal.first_bidder)
        self.trump        = None
        self.phase        = PHASE_BIDDING
        self.current_player = self.deal.first_bidder
        self.trick_number = 0
        self.current_trick = []
        self.trick_results = []
        self.cards_seen   = set()

    # ═══════════════════════════════════════════════════════════════════
    # Internal: advance through opponent turns until agent's turn
    # ═══════════════════════════════════════════════════════════════════

    def _advance_to_agent(self) -> tuple[np.ndarray, dict]:
        """
        Run opponent actions until the learning agent must act,
        or until the hand/game ends.
        Returns (obs, info) from the agent's perspective.
        """
        reward = 0.0

        while True:
            if self.game.game_over:
                break

            if self.phase == PHASE_SCORING:
                r = self._do_scoring()
                reward += r
                if self.game.game_over:
                    break
                # Start next hand
                self._start_new_hand(random.Random())
                continue

            if self.current_player == self.agent_player:
                break

            # Opponent acts
            legal   = self._legal_actions()
            obs     = self._get_observation()
            action  = self.opponent_policy(
                legal, obs=obs, phase=self.phase)
            self._apply_action(self.current_player, action)

        obs  = self._get_observation()
        info = {
            "legal_actions" : self._legal_actions(),
            "phase"         : self.phase,
            "trick_number"  : self.trick_number,
            "team_scores"   : dict(self.game.team_scores),
            "reward"        : reward,
        }
        return obs, info

    # ═══════════════════════════════════════════════════════════════════
    # Internal: apply one action for one player
    # ═══════════════════════════════════════════════════════════════════

    def _apply_action(self, player: int, action: int):
        if self.phase == PHASE_BIDDING:
            self._apply_bid(player, action)
        elif self.phase == PHASE_DECLARING:
            self._apply_declare(player, action)
        elif self.phase == PHASE_PLAYING:
            self._apply_play(player, action)

    def _apply_bid(self, player: int, action: int):
        bid_amount = action_to_bid(action)
        self.bid.place_bid(player, bid_amount)

        if self.bid.is_complete:
            if self.bid.is_misdeal:
                # Redeal same hand; dealer doesn't rotate on misdeal
                self._start_new_hand(random.Random())
            else:
                self.phase          = PHASE_DECLARING
                self.current_player = self.bid.bid_winner
        else:
            self.current_player = self.bid.current_player

    def _apply_declare(self, player: int, action: int):
        trump_suit  = action_to_suit(action)
        self.trump  = declare_trump(player, trump_suit)
        self.phase  = PHASE_PLAYING

        # First trick starts with the two kitty cards already in it,
        # then the bid winner leads the first actual card.
        self.current_trick = [
            (c, -1) for c in self.deal.kitty   # -1 = kitty (no player)
        ]
        self.current_player = self.bid.bid_winner

    def _apply_play(self, player: int, action: int):
        card = INDEX_TO_CARD[action]
        self.deal.hands[player].remove(card)
        self.current_trick.append((card, player))

        # A trick is complete when all 4 players have contributed.
        # Trick 1 already has 2 kitty cards, so only 4 player cards needed;
        # subsequent tricks need exactly 4.
        players_in_trick = sum(1 for _, p in self.current_trick if p >= 0)

        if players_in_trick == NUM_PLAYERS:
            self._resolve_trick()
        else:
            # Advance to next player
            self.current_player = (player + 1) % NUM_PLAYERS

    def _resolve_trick(self):
        trump_suit = self.trump.trump_suit if self.trump else None

        # Compute winner (kitty cards participate fully)
        winner_card, winner_player = trick_winner(
            self.current_trick, trump_suit)

        # winner_player == -1 means a kitty card won; award to bid winner
        if winner_player < 0:
            winner_player = self.bid.bid_winner

        cards_in_trick = [c for c, _ in self.current_trick]
        tr = make_trick_result(self.trick_number, winner_player, cards_in_trick)
        self.trick_results.append(tr)

        # Mark all non-kitty cards as seen
        for c, p in self.current_trick:
            if p >= 0:
                self.cards_seen.add(c.index)

        self.trick_number  += 1
        self.current_trick  = []

        if self.trick_number == 12:
            self.phase = PHASE_SCORING
        else:
            self.current_player = winner_player

    def _do_scoring(self) -> float:
        """Score the hand, update game state, return reward for agent."""
        result = score_hand(
            self.trick_results,
            bid_value  = self.bid.winning_bid,
            bid_winner = self.bid.bid_winner,
        )
        self.game.apply_hand_result(result)

        agent_team  = PLAYER_TEAM[self.agent_player]
        reward      = float(result.team_delta[agent_team])
        self.phase  = PHASE_BIDDING   # will be overridden by next hand start
        return reward

    # ═══════════════════════════════════════════════════════════════════
    # Internal: legal actions
    # ═══════════════════════════════════════════════════════════════════

    def _legal_actions(self) -> list[int]:
        if self.phase == PHASE_BIDDING:
            return [bid_to_action(b) for b in self.bid.legal_bids()]

        if self.phase == PHASE_DECLARING:
            # Bid winner may name any suit or no-trump
            return list(range(len(VALID_TRUMP_SUITS) + 1))   # 0-7

        if self.phase == PHASE_PLAYING:
            trump_suit   = self.trump.trump_suit if self.trump else None
            hand         = self.deal.hands[self.current_player]
            led_card     = self._led_card()
            is_first     = self.trick_number == 0
            is_leading   = led_card is None

            plays = legal_plays(
                hand          = hand,
                led_card      = led_card,
                trump_suit    = trump_suit,
                is_first_trick= is_first,
                is_leading    = is_leading,
            )
            return [c.index for c in plays]

        return []

    def _led_card(self) -> Optional[Card]:
        """
        The card that sets the led suit for following purposes.
        In trick 1, kitty cards are pre-loaded but don't set the led suit —
        the first card played by a human/agent sets it.
        """
        for card, player in self.current_trick:
            if player >= 0:
                return card
        return None

    # ═══════════════════════════════════════════════════════════════════
    # Internal: observation
    # ═══════════════════════════════════════════════════════════════════

    def _get_observation(self) -> np.ndarray:
        obs = np.zeros(OBS_TOTAL, dtype=np.float32)
        ap  = self.agent_player
        g   = self.game

        # My hand
        for c in self.deal.hands[ap]:
            obs[OBS_HAND_START + c.index] = 1.0

        # Kitty (always visible)
        for c in self.deal.kitty:
            obs[OBS_KITTY_START + c.index] = 1.0

        # Cards seen in prior tricks
        for idx in self.cards_seen:
            obs[OBS_SEEN_START + idx] = 1.0

        # Cards in current trick (including kitty in trick 1)
        for c, _ in self.current_trick:
            obs[OBS_TRICK_START + c.index] = 1.0

        # Trump suit one-hot (index 7 = no-trump)
        if self.trump is not None:
            if self.trump.no_trump:
                obs[OBS_TRUMP_START + 7] = 1.0
            else:
                suit_idx = VALID_TRUMP_SUITS.index(self.trump.trump_suit)
                obs[OBS_TRUMP_START + suit_idx] = 1.0

        # Current high bid (normalised)
        obs[OBS_HIGH_BID] = self.bid.current_bid / MAX_BID

        # Pass flags
        for p, amt in self.bid.bid_history:
            if amt == PASS:
                obs[OBS_PASS_FLAGS + p] = 1.0

        # Team scores (normalised, clipped to [-1, 1])
        for t in range(2):
            raw = g.team_scores[t] / WINNING_SCORE
            obs[OBS_SCORES + t] = np.clip(raw, -1.0, 1.0)

        # Whose turn (one-hot)
        obs[OBS_WHOSE_TURN + self.current_player] = 1.0

        # Phase (one-hot)
        obs[OBS_PHASE + self.phase] = 1.0

        return obs


# ═══════════════════════════════════════════════════════════════════════════════
# Quick smoke test
# ═══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    import warnings
    warnings.filterwarnings("ignore")   # suppress bonus-point warnings in demo

    print("SomeRSet Gymnasium environment — smoke test")
    print("=" * 55)

    env = SomeRSetEnv(agent_player=0, render_mode="human")
    obs, info = env.reset(seed=42)

    print(f"\nObs shape  : {obs.shape}")
    print(f"Action space: {env.action_space}")
    print(f"Legal actions at start: {info['legal_actions']}")

    # Run one full game with the agent also playing randomly
    total_hands = 0
    total_steps = 0
    episode_reward = 0.0

    obs, info = env.reset(seed=42)

    while not env.game.game_over:
        legal   = info["legal_actions"]
        action  = random.choice(legal)
        obs, reward, terminated, truncated, info = env.step(action)
        episode_reward += reward
        total_steps    += 1
        if terminated:
            break

    total_hands = env.game.hand_number
    print(f"\n{'='*55}")
    print(f"Game complete!")
    print(f"  Hands played  : {total_hands}")
    print(f"  Steps taken   : {total_steps}")
    print(f"  Winning team  : Team {env.game.winning_team}")
    print(f"  Final scores  : Team 0 = {env.game.team_scores[0]:+d}, "
          f"Team 1 = {env.game.team_scores[1]:+d}")
    print(f"  Agent reward  : {episode_reward:+.1f}")

    # Verify observation space compliance
    obs, info = env.reset(seed=99)
    assert env.observation_space.contains(obs), "Obs out of bounds!"
    print(f"\nObservation space check: PASSED")
    print(f"Obs vector sample (first 30 dims): {obs[:30]}")


# Training

In [ ]:
import argparse
import os
import warnings
import numpy as np
import gymnasium as gym
from gymnasium import spaces

from sb3_contrib import MaskablePPO
from sb3_contrib.common.maskable.callbacks import MaskableEvalCallback
from sb3_contrib.common.maskable.utils import get_action_masks
from sb3_contrib.common.wrappers import ActionMasker
from stable_baselines3.common.vec_env import SubprocVecEnv, VecMonitor
from stable_baselines3.common.callbacks import (
    CallbackList, CheckpointCallback, BaseCallback
)
from stable_baselines3.common.monitor import Monitor
from google.colab import drive

warnings.filterwarnings("ignore")

# Mount Drive
drive.mount('/content/drive')

# Create a folder in your Drive to save models
save_path = "/content/drive/MyDrive/Some-R-Set/"
os.makedirs(save_path, exist_ok=True)


# ── 1. Action-mask wrapper ─────────────────────────────────────────────────────
# MaskablePPO needs a callable mask_fn(env) → bool array of shape (n_actions,)
# We attach the mask to the env via ActionMasker.

def get_mask(env: SomeRSetEnv) -> np.ndarray:
    """
    Return a boolean mask over the full action space.
    True  = this action is legal right now.
    False = illegal; MaskablePPO will never sample it.
    """
    legal = env.legal_actions   # set by the wrapper property below
    mask  = np.zeros(env.action_space.n, dtype=bool)
    for a in legal:
        mask[a] = True
    return mask


class SomeRSetMaskedEnv(SomeRSetEnv):
    """
    Thin subclass that:
      1. Exposes a `legal_actions` property (required by get_mask).
      2. Wraps info["legal_actions"] so ActionMasker can find it.
    """
    @property
    def legal_actions(self) -> list[int]:
        return self._legal_actions()

    def action_masks(self) -> np.ndarray:
        """
        Called directly by MaskablePPO at every step.
        """
        return get_mask(self)

    # Override step to handle post-termination calls gracefully for stable-baselines3 evaluation
    def step(self, action: int):
        # If the game is already over (from a previous step), immediately return
        # a terminated state to allow the evaluation loop to call reset().
        if self.game.game_over:
            info_on_game_over = {
                "legal_actions": [], # No legal actions if game is truly over
                "phase": self.phase, # Current phase
                "trick_number": self.trick_number,
                "team_scores": dict(self.game.team_scores),
                "reward": 0.0, # The final reward was already reported by the *terminating* step
            }
            return self._get_observation(), 0.0, True, False, info_on_game_over

        # Original logic from SomeRSetEnv.step()
        # The original assertion `assert not self.game.game_over` is now handled by the above check.

        legal = self._legal_actions()
        if action not in legal:
            # Penalise illegal action and return current state unchanged
            obs  = self._get_observation()
            info = {"legal_actions": legal, "illegal_action": True}
            return obs, -1.0, False, False, info

        self._apply_action(self.agent_player, action)

        # Let opponents respond until it's the agent's turn
        obs, info = self._advance_to_agent()

        reward     = info.pop("reward", 0.0)
        terminated = self.game.game_over
        truncated  = False

        if self.render_mode == "human":
            self.render()

        return obs, reward, terminated, truncated, info


# ── 2. Environment factory ─────────────────────────────────────────────────────

def make_env(rank: int, seed: int = 0):
    """Factory function for SubprocVecEnv — each worker gets its own env."""
    def _init():
        try:
            # Seed the global random module for this worker process
            import random
            random.seed(seed + rank)

            env = SomeRSetMaskedEnv(agent_player=0)
            env = Monitor(env)          # records episode rewards/lengths
            env.reset(seed=seed + rank)
            return env
        except Exception as e:
            import traceback
            print(f"Error in worker {rank}: {e}")
            traceback.print_exc()
            # To ensure the error output is visible immediately
            import sys
            sys.stderr.flush()
            sys.stdout.flush()
            raise # Re-raise the exception after printing
    return _init


# ── 3. Custom training callback ────────────────────────────────────────────────

class SomeRSetCallback(BaseCallback):
    """
    Logs game-specific metrics to TensorBoard every N rollouts:
      - Mean episode reward
      - Win rate (agent's team reached 66 first)
      - Mean hands per game
      - Mean bid value
    """
    def __init__(self, log_freq: int = 10, verbose: int = 0):
        super().__init__(verbose)
        self.log_freq   = log_freq
        self.n_rollouts = 0

    def _on_rollout_end(self) -> bool:
        self.n_rollouts += 1
        if self.n_rollouts % self.log_freq != 0:
            return True

        # Pull episode info from the Monitor wrapper
        ep_infos = self.model.ep_info_buffer
        if not ep_infos:
            return True

        rewards = [ep["r"] for ep in ep_infos]
        lengths = [ep["l"] for ep in ep_infos]

        self.logger.record("somerset/mean_ep_reward", np.mean(rewards))
        self.logger.record("somerset/mean_ep_length", np.mean(lengths))

        if self.verbose >= 1:
            print(f"  Rollout {self.n_rollouts:4d} | "
                  f"mean_reward={np.mean(rewards):+.1f} | "
                  f"mean_ep_len={np.mean(lengths):.0f}")
        return True

    def _on_step(self) -> bool:
        """Called by the model every step. Return True to continue training."""
        return True


# ── 4. Evaluation helper ───────────────────────────────────────────────────────

def evaluate(model: MaskablePPO, n_episodes: int = 50) -> dict:
    """
    Run n_episodes with the trained model (deterministic=True) and return stats.
    Returns dict with keys: mean_reward, win_rate, mean_hands.
    """
    env = SomeRSetMaskedEnv(agent_player=0)
    rewards, wins, hands = [], [], []

    for ep in range(n_episodes):
        obs, info = env.reset(seed=1000 + ep)
        ep_reward = 0.0
        done      = False

        while not done:
            masks  = env.action_masks()
            action, _ = model.predict(
                obs, action_masks=masks, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(int(action))
            ep_reward += reward
            done = terminated or truncated

        rewards.append(ep_reward)
        hands.append(env.game.hand_number)
        # Win = agent's team (0) reached 66 first
        wins.append(1 if env.game.winning_team == 0 else 0)

    return {
        "mean_reward" : np.mean(rewards),
        "std_reward"  : np.std(rewards),
        "win_rate"    : np.mean(wins),
        "mean_hands"  : np.mean(hands),
    }


# ── 5. Main training routine ───────────────────────────────────────────────────

def train(
    total_steps : int   = 500_000,
    n_envs      : int   = 4,
    seed        : int   = 42,
    log_dir     : str   = "./logs/somerset",
    model_dir   : str   = "./models/somerset",
    eval_freq   : int   = 20_000,
    checkpoint_freq: int = 50_000,
    n_eval_eps  : int   = 20,
    learning_rate: float = 3e-4,
    n_steps     : int   = 2048,    # rollout steps per env before update
    batch_size  : int   = 256,
    n_epochs    : int   = 10,
    gamma       : float = 0.99,
    verbose     : int   = 1,
):
    os.makedirs(log_dir,   exist_ok=True)
    os.makedirs(model_dir, exist_ok=True)

    # ── Vectorised training envs ──────────────────────────────────────
    print(f"Creating {n_envs} parallel training environments...")
    vec_env = SubprocVecEnv([make_env(i, seed) for i in range(n_envs)])
    vec_env = VecMonitor(vec_env, filename=os.path.join(log_dir, "monitor"))

    # ── Single eval env ───────────────────────────────────────────────
    eval_env_raw = SomeRSetMaskedEnv(agent_player=0)
    eval_env     = Monitor(eval_env_raw,
                           filename=os.path.join(log_dir, "eval_monitor"))

    # ── Model ─────────────────────────────────────────────────────────
    # Policy network: MlpPolicy with two hidden layers.
    # For card games, a larger network (256×256 or 512×512) often helps
    # because the agent must learn complex suit/trump interactions.
    policy_kwargs = dict(
        net_arch=[256, 256],   # two hidden layers of 256 units each
    )

    model = MaskablePPO(
        policy          = "MlpPolicy",
        env             = vec_env,
        learning_rate   = learning_rate,
        n_steps         = n_steps,
        batch_size      = batch_size,
        n_epochs        = n_epochs,
        gamma           = gamma,
        gae_lambda      = 0.95,
        clip_range      = 0.2,
        ent_coef        = 0.01,     # entropy bonus encourages exploration
        vf_coef         = 0.5,
        max_grad_norm   = 0.5,
        policy_kwargs   = policy_kwargs,
        tensorboard_log = log_dir,
        seed            = seed,
        verbose         = verbose,
    )

    print(f"Model parameters: {sum(p.numel() for p in model.policy.parameters()):,}")
    print(f"Observation size: {vec_env.observation_space.shape}")
    print(f"Action size:      {vec_env.action_space.n}")

    # ── Callbacks ─────────────────────────────────────────────────────
    checkpoint_cb = CheckpointCallback(
        save_freq   = max(checkpoint_freq // n_envs, 1),
        save_path   = model_dir,
        name_prefix = "somerset_ppo",
    )

    eval_cb = MaskableEvalCallback(
        eval_env,
        best_model_save_path = model_dir,
        log_path             = log_dir,
        eval_freq            = max(eval_freq // n_envs, 1),
        n_eval_episodes      = n_eval_eps,
        deterministic        = True,
        render               = False,
    )

    log_cb = SomeRSetCallback(log_freq=5, verbose=verbose)

    callbacks = CallbackList([checkpoint_cb, eval_cb, log_cb])

    # ── Train ─────────────────────────────────────────────────────────
    print(f"\nTraining for {total_steps:,} steps across {n_envs} envs...")
    print(f"Logs  → {log_dir}")
    print(f"Models→ {model_dir}")
    print(f"Run:  tensorboard --logdir {log_dir}\n")

    model.learn(
        total_timesteps     = total_steps,
        callback             = callbacks,
        reset_num_timesteps = True,
        progress_bar        = True,
    )

    # ── Save final model ──────────────────────────────────────────────
    final_path = os.path.join(model_dir, "somerset_ppo_final")
    model.save(final_path)
    print(f"\nFinal model saved → {final_path}.zip")

    # ── Evaluate ──────────────────────────────────────────────────────
    print(f"\nEvaluating over {n_eval_eps} episodes...")
    stats = evaluate(model, n_episodes=n_eval_eps)
    print(f"  Mean reward : {stats['mean_reward']:.1f} \u00b1 {stats['std_reward']:.1f}")
    print(f"  Win rate    : {stats['win_rate']*100:.1f}%")
    print(f"  Mean hands  : {stats['mean_hands']:.1f}")

    vec_env.close()
    eval_env.close()
    return model, stats

if __name__ == "__main__":
  train(
      total_steps   = 100000,
      n_envs         = 4,
      seed          = 42,
      learning_rate = 3e-4,
      eval_freq     = 4000,
      log_dir       = save_path + "logs",
      model_dir     = save_path + "models",
  )

In [ ]:
model = MaskablePPO.load(os.path.join(save_path, "models","somerset_ppo_final.zip"))
env   = SomeRSetMaskedEnv(
    agent_player=0,
    render_mode="human",
)

n_episodes = 1

for ep in range(n_episodes):
    obs, info = env.reset(seed=ep)
    ep_reward = 0.0
    print(f"\n{'═'*55}")
    print(f"Episode {ep+1}")

    while True:
        masks  = env.action_masks()
        action, _ = model.predict(obs, action_masks=masks, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(int(action))
        ep_reward += reward
        if terminated or truncated:
            break

    result = "WIN" if env.game.winning_team == 0 else "LOSS"
    print(f"\n{result} | Reward: {ep_reward:+.1f} | "
          f"Hands: {env.game.hand_number} | "
          f"Final: {env.game.team_scores}")